# 05 — Ideal-state QAOA

Runs QAOA on the compact path-QUBO benchmarks with exact classical optima and zero path-reduction gap.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa.benchmarks import (
    FROZEN_QAOA_BENCHMARKS,
    build_frozen_benchmark,
    physical_feasibility_mask,
)
from oec_qaoa.qaoa import optimize_qaoa

DEPTHS = [1, 2, 3, 4]
RESTART_SEEDS = [101, 202, 303, 404, 505]
MAXITER = 200

## Benchmark set

The selected benchmarks use positive ground-processing objectives and path reductions with `Δ_K = 0`. They span 16–20 logical variables.

In [ ]:
benchmark_rows = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, retained, model = build_frozen_benchmark(benchmark)
    benchmark_rows.append(
        {
            "benchmark_id": benchmark.benchmark_id,
            "instance": instance.name,
            "K": benchmark.k,
            "logical_variables": len(model.variable_names),
            "semantic_variables": model.semantic_count,
            "quadratic_couplings": len(model.quadratic),
        }
    )

pd.DataFrame(benchmark_rows)

## Variational protocol

For depth `p`, QAOA uses the standard transverse-field mixer and the explicit diagonal path-QUBO cost Hamiltonian. Each depth `p = 1,2,3,4` is optimized with COBYLA from five deterministic initializations, with a 200-evaluation budget.

In [ ]:
records = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, retained, model = build_frozen_benchmark(benchmark)
    feasible_mask = physical_feasibility_mask(instance, retained, model)

    for depth in DEPTHS:
        for restart, seed in enumerate(RESTART_SEEDS, start=1):
            result = optimize_qaoa(
                model,
                depth=depth,
                seed=seed,
                feasible_mask=feasible_mask,
                optimizer="COBYLA",
                maxiter=MAXITER,
            )

            records.append(
                {
                    "benchmark_id": benchmark.benchmark_id,
                    "depth": depth,
                    "restart": restart,
                    "seed": seed,
                    "expected_energy": result.expected_energy,
                    "optimal_energy": result.optimal_energy,
                    "expected_gap": (
                        result.expected_energy - result.optimal_energy
                    ),
                    "optimal_probability": result.optimal_probability,
                    "feasible_probability": result.feasible_probability,
                    "most_likely_energy": result.most_likely_energy,
                    "n_evaluations": result.n_evaluations,
                    "optimizer_success": result.success,
                }
            )

runs = pd.DataFrame(records)
runs.head()

## Aggregated results

The summary reports mean and best expected energy, optimal-state probability, feasible-state probability and optimizer evaluations across restarts.

In [ ]:
summary = (
    runs.groupby(["benchmark_id", "depth"])
    .agg(
        expected_energy_mean=("expected_energy", "mean"),
        expected_energy_best=("expected_energy", "min"),
        expected_gap_mean=("expected_gap", "mean"),
        optimal_probability_mean=("optimal_probability", "mean"),
        optimal_probability_best=("optimal_probability", "max"),
        feasible_probability_mean=("feasible_probability", "mean"),
        feasible_probability_best=("feasible_probability", "max"),
        n_evaluations_mean=("n_evaluations", "mean"),
    )
    .reset_index()
)

summary

## Result summary

The normalized cost Hamiltonian improves numerical conditioning. Increasing depth enlarges the variational family, but the optimized performance is not monotonic in `p` under the fixed classical optimization budget.